In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        filepath=os.path.join(dirname, filename)

In [ ]:
df = pd.read_csv(filepath)
df.head()

# EDA

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sum()  # Identifying null value = O nulls
df.duplicated().sum()   # Finding Duplicated = 0 Duplicates
df.shape
unique_entries = [df[col].unique() for col in df.columns]
df['Price (Inr)'] = df['Price (Euro)']*109.86

In [ ]:
df.columns

In [ ]:
import seaborn as sns
sns.displot(df['Price (Euro)'],kde=True)

In [ ]:
df['Company'].value_counts().plot(kind='bar')

In [ ]:
df['OpSys'].value_counts().plot(kind='bar')

In [ ]:
df['GPU_Company'].value_counts().plot(kind='bar')

In [ ]:
import matplotlib.pyplot as plt
sns.barplot(df,x="Company",y="Price (Inr)",color='red')
plt.xticks(rotation="vertical")
plt.show()

In [ ]:
# mean_prices = df.groupby('4k')['Price (Inr)'].mean()
# pct_increase = ((mean_prices[1] - mean_prices[0]) / mean_prices[0]) * 100

# sns.barplot(data=df, x='4k', y='Price (Inr)', color='yellow')
# plt.xticks([0, 1], ['Non-4K', '4K'])
# plt.text(
#     0.5, 
#     max(mean_prices) * 0.95, 
#     f"+{pct_increase:.1f}%", 
#     ha='center', 
#     fontsize=14, 
#     color='green',
#     fontweight='bold'
# )

# plt.title("Price Comparison: Non-4K vs 4K")
# plt.show()

In [ ]:
 # df.corr()['Price (Inr)']
df.select_dtypes(["int64","float64"]).corr()['Price (Inr)']

In [ ]:
def cpu_tier(cpu):
    if 'Xeon' in cpu or 'i7' in cpu and ('HQ' in cpu or 'HK' in cpu):
        return 'high'
    elif 'i7' in cpu or 'Ryzen' in cpu or 'FX' in cpu:
        return 'mid_high'
    elif 'i5' in cpu:
        return 'mid'
    else:
        return 'low'
def gpu_tier(gpu):
    if any(x in gpu for x in ['GTX 1080','GTX 1070','GTX 1060','980','970','Quadro','FirePro']):
        return 'high'
    elif any(x in gpu for x in ['GTX 1050','GTX 960','RX 580','RX 560']):
        return 'mid_high'
    elif any(x in gpu for x in ['MX150','MX130','940','930','920','Radeon 530','Radeon 540']):
        return 'mid'
    else:
        return 'low'

def os_identifier(os):
    if os in ["macOS","Mac OS X"]:
        return 'macOS'
    elif os in ['Windows 10','Windows 10 S','Windows 7']:
        return 'Windows'
    else:
        return "Unpopular os"
def extract_series(name):
    keywords = ['MacBook','XPS','ThinkPad','ROG','Legion','Inspiron','Pavilion','EliteBook','ProBook','Aspire','ZenBook','VivoBook','Omen','Alienware']
    for k in keywords:
        if k.lower() in name.lower():
            return k
    return 'Other'

In [ ]:
df['GPU_Tier'] = df['GPU_Type'].apply(lambda x:gpu_tier(x))
df['CPU_Tier'] = df['CPU_Type'].apply(lambda x:cpu_tier(x))
df['OpSys'] = df['OpSys'].apply(lambda x:os_identifier(x))
df['Product'] = df['Product'].apply(lambda x:extract_series(x))

In [ ]:
premium = ['MacBook','XPS','ThinkPad','EliteBook','Spectre','Alienware','ROG']
df['is_premium'] = df['Product'].apply(lambda x: 1 if x in premium else 0)

In [ ]:
df.head()

In [ ]:
df['Memory'].unique()

In [ ]:
import re

def process_memory(mem):
    ssd = 0
    hdd = 0
    flash = 0
    
    parts = mem.split('+')
    
    for part in parts:
        part = part.strip()
        
        size = int(re.findall(r'\d+', part)[0])
        
        if 'TB' in part:
            size *= 1024
        if 'SSD' in part:
            ssd += size
        elif 'HDD' in part:
            hdd += size
        elif 'Flash' in part:
            flash += size
    
    return ssd, hdd, flash
df[['SSD','HDD','Flash']] = df['Memory'].apply(lambda x: pd.Series(process_memory(x)))

In [ ]:
df.drop(columns=['Inches','Weight (kg)','Price (Euro)','CPU_Type','GPU_Type','ScreenResolution','Memory','Product'],inplace=True)

In [ ]:
df.select_dtypes(["int64","float64"]).corr()['Price (Inr)']

In [ ]:
df['Company'].value_counts()
company_mean_price = df.groupby('Company')['Price (Inr)'].mean()
sns.barplot(x=company_mean_price.index,y=company_mean_price.values)
plt.xticks(rotation="vertical")
plt.show()
print(df['Company'].value_counts())

In [ ]:
def other_companies(companies):
    if companies in ['Vero','Xiaomi','Chuwi','Fujitsu','LG','Huawei']:
        return "other_companies"
    else:
        return companies
df['Company'] = df['Company'].apply(lambda x:pd.Series(other_companies(x)))

In [ ]:
[f"{x} => {df[x].unique()}" for x in df.select_dtypes("object").columns]

# Training Phase

In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop(columns =['Price (Inr)'])
y = df['Price (Inr)']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.15,random_state=2)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder
from sklearn.metrics import r2_score,mean_absolute_error
from sklearn.preprocessing import StandardScaler


from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor,AdaBoostRegressor,ExtraTreesRegressor,VotingRegressor,StackingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor

In [ ]:
print(X_train.shape)
X_train.head()

In [ ]:
# LinearRegression
step1 = ColumnTransformer(
    transformers=[
        ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  LinearRegression()
pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
lrr2 = r2_score(y_test,y_pred)
lrmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# Ridge
step1 = ColumnTransformer(
    transformers=[
        ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  Ridge()
pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
rir2 = r2_score(y_test,y_pred)
rimae = mean_absolute_error(y_test,y_pred)

In [ ]:
# Lasso
step1 = ColumnTransformer(
    transformers=[
        ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  Lasso()
pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
lar2 = r2_score(y_test,y_pred)
lamae = mean_absolute_error(y_test,y_pred)

In [ ]:
# KNeighborsRegressor
step1 = ColumnTransformer(
    transformers=[
        ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  KNeighborsRegressor()
pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
knr2 = r2_score(y_test,y_pred)
knmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# DecisionTreeRegressor
step1 = ColumnTransformer(
    transformers=[
        # ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  DecisionTreeRegressor()
pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
dtr2 = r2_score(y_test,y_pred)
dtmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# RandomForestRegressor
step1 = ColumnTransformer(
    transformers=[
        # ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  RandomForestRegressor()
pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
rfr2 = r2_score(y_test,y_pred)
rfmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# GradientBoostingRegressor
step1 = ColumnTransformer(
    transformers=[
        # ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  GradientBoostingRegressor()
pipegb = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
gbr2 = r2_score(y_test,y_pred)
gbmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# AdaBoostRegressor
step1 = ColumnTransformer(   
    transformers=[
        # ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  AdaBoostRegressor()
pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
agr2 = r2_score(y_test,y_pred)
agmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# ExtraTreesRegressor
step1 = ColumnTransformer(
    transformers=[
        # ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  ExtraTreesRegressor()
pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
etr2 = r2_score(y_test,y_pred)
etmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# VotingRegressor
step1 = ColumnTransformer(
    transformers=[
        ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
rf = RandomForestRegressor(n_estimators=350,random_state=3,max_samples=0.5,max_features=0.75,max_depth=15)
gbdt = GradientBoostingRegressor(n_estimators=100,max_features=0.5)
xgb = XGBRegressor(n_estimators=25,learning_rate=0.3,max_depth=5)
et = ExtraTreesRegressor(n_estimators=100,random_state=3,max_samples=0.5,max_features=0.75,max_depth=10, bootstrap=True)

step2 = VotingRegressor([('rf', rf), ('gbdt', gbdt), ('xgb',xgb), ('et',et)],weights=[5,1,1,1])

pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
vrr2 = r2_score(y_test,y_pred)
vrmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# svr
step1 = ColumnTransformer(
    transformers=[
        ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  SVR()
pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
svr2 = r2_score(y_test,y_pred)
svmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# XGBRegressor
step1 = ColumnTransformer(
    transformers=[
        ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
step2 =  XGBRegressor()
pipelr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipelr.fit(X_train,y_train)
y_pred = pipelr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
xgr2 = r2_score(y_test,y_pred)
xgmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# StackingRegressor
step1 = ColumnTransformer(
    transformers=[
        ('scaler',StandardScaler(),[3,4,9,10,11,12]),
        ('OneHotEncd',OneHotEncoder(drop='first'),[1,2,5,6]),
        ('OrdinalEncod',OrdinalEncoder(),[0,7,8]),
    ],remainder='passthrough'
)
estimators = [
    ('rf', RandomForestRegressor(n_estimators=350,random_state=3,max_samples=0.5,max_features=0.75,max_depth=15)),
    ('gbdt',GradientBoostingRegressor(n_estimators=100,max_features=0.5)),
    ('xgb', XGBRegressor(n_estimators=25,learning_rate=0.3,max_depth=5))
]

step2 = StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=100))
pipesr = Pipeline(
    [('step1',step1),
    ('step2',step2)]
)
pipesr.fit(X_train,y_train)
y_pred = pipesr.predict(X_test)

print('R2 score',r2_score(y_test,y_pred))
print('MAE',mean_absolute_error(y_test,y_pred))
srr2 = r2_score(y_test,y_pred)
srmae = mean_absolute_error(y_test,y_pred)

In [ ]:
# evaluation
print(f"linear regression : r2score {lrr2} , mae {lrmae}")
print(f"Ridge regression : r2score {rir2} , mae {rimae}")
print(f"lasso regression : r2score {lar2} , mae {lamae}")
print(f"Knn regression : r2score {knr2} , mae {knmae}")
print(f"Decision regression : r2score {dtr2} , mae {dtmae}")
print(f"Random forest regression : r2score {rfr2} , mae {rfmae}")
print(f"Gradient boosting regression : r2score {gbr2} , mae {gbmae}")
print(f"extra tree regression : r2score {etr2} , mae {etmae}")
print(f"ada boost regression : r2score {agr2} , mae {agmae}")
print(f"svr regression : r2score {svr2} , mae {svmae}")
print(f"xg regression : r2score {xgr2} , mae {xgmae}")
print(f"votign regression : r2score {vrr2} , mae {vrmae}")
print(f"StackingRegressor regression : r2score {srr2} , mae {srmae}")

In [ ]:
import joblib as jb
jb.dump(df,'df.joblib')
jb.dump(pipesr,'StackingRegressor.joblib')

In [10]:
df= pd.read_csv('/home/ayush/Documents/AI/Projects/MLOPS/Laptop_Price_Prediction_GCP_MLOPS/artifacts/ModelTrainingArtifacts/train_df.csv')
X_train = df.drop(columns =['Price (Inr)'])
y_train = df['Price (Inr)']

X_train.head()

,Unnamed: 0.1,Unnamed: 0,Company,TypeName,CPU_Company,CPU_Frequency (GHz),RAM (GB),GPU_Company,OpSys,GPU_Tier,CPU_Tier,is_premium,SSD,HDD,Flash
0,944,944,Lenovo,2 in 1 Convertible,Intel,1.2,8,Intel,Windows,low,low,0,256,0,0
1,21,21,Lenovo,Gaming,Intel,2.5,8,Nvidia,Windows,mid_high,mid,0,128,1024,0
2,530,530,Dell,Gaming,Intel,2.8,16,Nvidia,Windows,high,high,1,128,1024,0
3,279,279,Lenovo,Notebook,Intel,1.8,8,Nvidia,Unpopular os,mid,mid_high,0,0,2048,0
4,398,398,Dell,Workstation,Intel,2.8,8,Nvidia,Windows,high,high,0,256,0,0
